# Notebook 3: 优化与机器学习模型实现

本 Notebook 整合优化理论与机器学习实践，使用 Scikit-learn 实现完整 ML 流程。

## 1. 导入库

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import minimize

from sklearn.datasets import make_classification, make_regression, load_iris
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, mean_squared_error, r2_score

sns.set_style('whitegrid')
np.random.seed(42)

print("库导入完成!")

## 2. 梯度下降法实现

In [ ]:
def gradient_descent(f, grad_f, x0, alpha=0.01, max_iter=100, tol=1e-6):
    """
    梯度下降法
    
    参数:
        f: 目标函数
        grad_f: 梯度函数
        x0: 初始点
        alpha: 学习率
        max_iter: 最大迭代次数
        tol: 收敛容差
    
    返回:
        x: 最优解
        history: 迭代历史
    """
    x = np.array(x0, dtype=float)
    history = [x.copy()]
    
    for i in range(max_iter):
        grad = grad_f(x)
        if np.linalg.norm(grad) < tol:
            print(f"在迭代 {i} 收敛")
            break
        x = x - alpha * grad
        history.append(x.copy())
    
    return x, np.array(history)

# 测试函数：f(x) = x² - 4x + 5
def f_obj(x):
    return x[0]**2 - 4*x[0] + 5

def grad_f_obj(x):
    return np.array([2*x[0] - 4])

# 运行梯度下降
x_opt, history = gradient_descent(f_obj, grad_f_obj, x0=[10], alpha=0.1, max_iter=50)

print(f"优化结果:")
print(f"  最优解 x* = {x_opt[0]:.6f}")
print(f"  最小值 f(x*) = {f_obj(x_opt):.6f}")
print(f"  理论最优解：x = 2, f(2) = 1")

In [ ]:
# 可视化梯度下降过程
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 函数图像
x_plot = np.linspace(-2, 8, 100)
y_plot = x_plot**2 - 4*x_plot + 5

axes[0].plot(x_plot, y_plot, 'b-', label='f(x) = x² - 4x + 5', linewidth=2)
axes[0].plot(history[:, 0], [f_obj([x]) for x in history[:, 0]], 'ro-', 
             label='梯度下降路径', markersize=8)
axes[0].axvline(x=2, color='g', linestyle='--', label='最小值 x=2')
axes[0].set_xlabel('x')
axes[0].set_ylabel('f(x)')
axes[0].set_title('梯度下降可视化')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 收敛曲线
f_values = [f_obj([x]) for x in history[:, 0]]
axes[1].plot(range(len(f_values)), f_values, 'bo-', linewidth=2, markersize=5)
axes[1].set_xlabel('迭代次数')
axes[1].set_ylabel('f(x)')
axes[1].set_title('收敛曲线')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/home/admin/.openclaw/workspace/DataScience_Math_Python/notebooks/gradient_descent.png', dpi=150)
plt.show()

## 3. 使用 SciPy 优化

In [ ]:
# Rosenbrock 函数（经典优化测试函数）
def rosenbrock(x):
    return (1 - x[0])**2 + 100*(x[1] - x[0]**2)**2

# 使用 SciPy 优化
result = minimize(rosenbrock, [0, 0], method='BFGS')

print("Rosenbrock 函数优化:")
print(f"  函数：f(x,y) = (1-x)² + 100(y-x²)²")
print(f"  初始点：[0, 0]")
print(f"  最优解：{result.x}")
print(f"  最小值：{result.fun:.10f}")
print(f"  理论最优解：[1, 1], f = 0")
print(f"  成功：{result.success}")

## 4. 机器学习分类任务

In [ ]:
# 加载 Iris 数据集
iris = load_iris()
X, y = iris.data, iris.target

print("Iris 数据集:")
print(f"  样本数：{X.shape[0]}")
print(f"  特征数：{X.shape[1]}")
print(f"  特征名：{iris.feature_names}")
print(f"  类别：{iris.target_names}")

# 分割数据
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\n训练集：{X_train.shape}, 测试集：{X_test.shape}")

In [ ]:
# 标准化
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# 训练多个分类器并比较
classifiers = {
    '逻辑回归': LogisticRegression(max_iter=200, random_state=42),
    '决策树': DecisionTreeClassifier(max_depth=3, random_state=42),
    '随机森林': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(kernel='rbf', random_state=42),
    '梯度提升': GradientBoostingClassifier(n_estimators=100, random_state=42)
}

results = []

for name, clf in classifiers.items():
    clf.fit(X_train_scaled, y_train)
    y_pred = clf.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    cv_scores = cross_val_score(clf, X_train_scaled, y_train, cv=5)
    
    results.append({
        '模型': name,
        '测试准确率': acc,
        'CV 平均': cv_scores.mean(),
        'CV 标准差': cv_scores.std()
    })
    
    print(f"{name}: 测试准确率 = {acc:.4f}, CV = {cv_scores.mean():.4f} (+/- {cv_scores.std()*2:.4f})")

# 结果 DataFrame
results_df = pd.DataFrame(results)
print("\n模型比较:")
print(results_df.sort_values('测试准确率', ascending=False))

In [ ]:
# 最佳模型的混淆矩阵
best_model = RandomForestClassifier(n_estimators=100, random_state=42)
best_model.fit(X_train_scaled, y_train)
y_pred_best = best_model.predict(X_test_scaled)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 混淆矩阵
cm = confusion_matrix(y_test, y_pred_best)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=iris.target_names,
            yticklabels=iris.target_names)
axes[0].set_xlabel('预测')
axes[0].set_ylabel('真实')
axes[0].set_title('混淆矩阵')

# 特征重要性
feature_importance = pd.DataFrame({
    '特征': iris.feature_names,
    '重要性': best_model.feature_importances_
}).sort_values('重要性', ascending=False)

sns.barplot(data=feature_importance, x='重要性', y='特征', ax=axes[1], palette='viridis')
axes[1].set_title('特征重要性')

plt.tight_layout()
plt.savefig('/home/admin/.openclaw/workspace/DataScience_Math_Python/notebooks/ml_results.png', dpi=150)
plt.show()

print("\n分类报告:")
print(classification_report(y_test, y_pred_best, target_names=iris.target_names))

## 5. 超参数优化

In [ ]:
# 网格搜索
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 10, None],
    'min_samples_split': [2, 5, 10]
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train_scaled, y_train)

print("\n网格搜索结果:")
print(f"  最佳参数：{grid_search.best_params_}")
print(f"  最佳 CV 得分：{grid_search.best_score_:.4f}")
print(f"  测试集得分：{grid_search.score(X_test_scaled, y_test):.4f}")

In [ ]:
# 可视化网格搜索结果
cv_results = pd.DataFrame(grid_search.cv_results_)

# 提取 n_estimators 和 max_depth 的结果
pivot_data = cv_results.pivot_table(
    index='param_max_depth',
    columns='param_n_estimators',
    values='mean_test_score'
)

plt.figure(figsize=(10, 6))
sns.heatmap(pivot_data.astype(float), annot=True, fmt='.3f', cmap='YlOrRd')
plt.title('网格搜索结果热力图')
plt.xlabel('n_estimators')
plt.ylabel('max_depth')
plt.tight_layout()
plt.savefig('/home/admin/.openclaw/workspace/DataScience_Math_Python/notebooks/grid_search.png', dpi=150)
plt.show()

## 6. 回归任务

In [ ]:
# 生成回归数据
X_reg, y_reg = make_regression(n_samples=200, n_features=5, n_informative=3,
                                noise=10, random_state=42)

X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

# 训练回归模型
regressors = {
    '线性回归': LinearRegression(),
    '岭回归': Ridge(alpha=1.0),
    'Lasso 回归': Lasso(alpha=0.1),
    '随机森林回归': RandomForestRegressor(n_estimators=100, random_state=42)
}

from sklearn.ensemble import RandomForestRegressor

print("回归模型比较:")
for name, reg in regressors.items():
    reg.fit(X_reg_train, y_reg_train)
    y_pred_reg = reg.predict(X_reg_test)
    mse = mean_squared_error(y_reg_test, y_pred_reg)
    r2 = r2_score(y_reg_test, y_pred_reg)
    print(f"  {name}: MSE = {mse:.2f}, R² = {r2:.4f}")

## 7. 练习

In [ ]:
# 练习 1：实现多元梯度下降
def gradient_descent_2d(f, grad_f, x0, y0, alpha=0.1, n_iterations=20):
    """二维梯度下降"""
    x, y = x0, y0
    history = [(x, y, f(x, y))]
    
    for i in range(n_iterations):
        gx, gy = grad_f(x, y)
        x = x - alpha * gx
        y = y - alpha * gy
        history.append((x, y, f(x, y)))
    
    return history

# 目标函数：f(x, y) = x² + 2y²
def f_2d(x, y):
    return x**2 + 2*y**2

def grad_2d(x, y):
    return (2*x, 4*y)

history_2d = gradient_descent_2d(f_2d, grad_2d, x0=5, y0=5, alpha=0.1, n_iterations=15)

print("练习 1：二维梯度下降")
print(f"初始点：(5, 5), f = {f_2d(5, 5)}")
print(f"最终点：({history_2d[-1][0]:.6f}, {history_2d[-1][1]:.6f}), f = {history_2d[-1][2]:.6f}")
print(f"理论最小值：(0, 0), f = 0")

In [ ]:
# 练习 2：完整的 ML 流程
# 从数据生成到模型评估

# 1. 生成数据
X_ex, y_ex = make_classification(n_samples=500, n_features=10, n_informative=8,
                                  n_redundant=2, random_state=42)

# 2. 分割数据
X_ex_train, X_ex_test, y_ex_train, y_ex_test = train_test_split(
    X_ex, y_ex, test_size=0.2, random_state=42, stratify=y_ex
)

# 3. 标准化
scaler_ex = StandardScaler()
X_ex_train_scaled = scaler_ex.fit_transform(X_ex_train)
X_ex_test_scaled = scaler_ex.transform(X_ex_test)

# 4. 训练模型
clf_ex = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
clf_ex.fit(X_ex_train_scaled, y_ex_train)

# 5. 评估
y_ex_pred = clf_ex.predict(X_ex_test_scaled)
acc_ex = accuracy_score(y_ex_test, y_ex_pred)

print("\n练习 2：完整 ML 流程")
print(f"  数据集：{X_ex.shape}")
print(f"  训练集：{X_ex_train.shape}, 测试集：{X_ex_test.shape}")
print(f"  模型：随机森林 (n_estimators=100, max_depth=10)")
print(f"  测试准确率：{acc_ex:.4f}")
print(f"  特征重要性前 3: {clf_ex.feature_importances_.argsort()[-3:][::-1]}")

## 总结

本 Notebook 涵盖了：
- ✅ 梯度下降法实现与可视化
- ✅ SciPy 优化库使用
- ✅ 多种分类算法比较
- ✅ 混淆矩阵与特征重要性
- ✅ 网格搜索超参数优化
- ✅ 回归任务实现
- ✅ 完整机器学习流程